In [1]:
import torch
import logging
import numpy as np
from cellwhisperer.jointemb.cellwhisperer_lightning import TranscriptomeTextDualEncoderLightning
from cellwhisperer.jointemb.processing import TranscriptomeTextDualEncoderProcessor
from cellwhisperer.config import get_path, model_path_from_name

from cellwhisperer.jointemb.dataset.jointemb import JointEmbedDataModule


ModuleNotFoundError: No module named 'cellwhisperer'

In [ ]:
# configure logging
log_file = snakemake.log.log_file  # Get the log file path from Snakemake
logging.basicConfig(
    filename=log_file,
    filemode='a',  # Append to the existing log file
    level=logging.INFO,  # Set the logging level (e.g., DEBUG, INFO, WARNING, ERROR, CRITICAL)
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'  # Define the log message format
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pl_model = TranscriptomeTextDualEncoderLightning.load_from_checkpoint(snakemake.input.model)
pl_model.eval().to(device)
pl_model.model.prepare_models(
    pl_model.model.transcriptome_model, pl_model.model.text_model, force_freeze=True
)
pl_model.freeze()

In [1]:
# from tqdm.notebook import tqdm

# traverse all transcriptomes, log features and embeddings and save them

dm = JointEmbedDataModule(dataset_name=snakemake.wildcards["dataset"],
                     transcriptome_processor=pl_model.model.transcriptome_model.config.model_type,
                     tokenizer=pl_model.model.text_model.config.model_type,
                     batch_size=32,
                     min_genes=0,
                     train_fraction=0.0)  # use val dataloader to prevent shuffling of datapoints

dm.prepare_data(force_prepare=False)  # Set to true to force recomputation
dm.setup()
dl = dm.val_dataloader()
results = []

for batch in dm.val_dataloader():  # tqdm(, desc=f"Processing {snakemake.wildcards['dataset']}"):
    result = pl_model(**{k: t.to(device) for k, t in batch.items()})
    result["transcriptome_text_similarity"] = torch.diag(result["logits_per_transcriptome"])
    results.append({k: t.detach().cpu() for k, t in result.items() if k not in ["logits_per_transcriptome", "logits_per_text"]})

NameError: name 'JointEmbedDataModule' is not defined

In [ ]:
aggregated_dict = {key: torch.cat([d[key] for d in results]) for key in results[0]}
aggregated_dict["orig_ids"] = dl.dataset.orig_ids
assert len(aggregated_dict["orig_ids"]) == len(aggregated_dict["transcriptome_text_similarity"])

In [ ]:
np.savez(snakemake.output["model_outputs"], **aggregated_dict) 